# Phase 2 Legal NER — 6 additional fine-tuning methods
Houlsby adapters · Pfeiffer adapters · Prefix tuning · Prompt tuning · BitFit · Knowledge distillation (+ logit ensemble).
Same Phase-1 `EfficientLegalLM`, same BIO tagger / CRF / bidirectional masks / windowed inference / strict-F1 pipeline as the first notebook. Each method writes `/kaggle/working/submission_<method>.zip`.

In [1]:
import os, sys, json, math, random, time, zipfile, glob, re, bisect
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Logged in.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in.


## Config

In [2]:
# ----------------------------- which methods to run -----------------------------
METHODS_TO_RUN = ["houlsby", "pfeiffer", "prefix", "prompt", "bitfit", "kd"]   # run any subset / order
RUN_ENSEMBLE   = True     # after the loop: logit-average every finished method -> submission_ensemble.zip

# ----------------------------- shared switches (same meaning as your first notebook) ----
USE_CRF        = True
BIDIRECTIONAL  = True
BIDIR_WINDOW   = 256

# ----------------------------- method hyper-parameters --------------------------
ADAPTER_R      = {"houlsby": 32, "pfeiffer": 64}   # bottleneck width (Houlsby has 2 adapters/layer, Pfeiffer 1 -> give it more)
N_PROMPT       = 32       # soft-prompt tokens prepended to every window     (prompt)
N_PREFIX       = 32       # learnable K/V prefix slots in every layer         (prefix)
# KD: student = a subset of the (fine-tuned) teacher's layers; sliding/full alternation of the kept layers is preserved
KD_KEEP        = [0, 1, 4, 5, 8, 9, 12, 13, 18, 19]
KD_TEMP, KD_ALPHA = 2.0, 0.5       # loss = (1-a)*hard + a*T^2*KL(teacher||student) on the tag emissions
TEACHER_CKPT   = None     # e.g. "/kaggle/working/lner_full_crf_bi/best_lner_full.pt" from your first notebook; None -> train a teacher here

# ----------------------------- data / windows -----------------------------------
WIDTH, STRIDE  = 512, 256
DEV_COUNT      = 8
INFER_BATCH    = 16

# ----------------------------- optimisation ------------------------------------
BATCH          = 8
EPOCHS_BY      = {"houlsby": 10, "pfeiffer": 10, "prefix": 12, "prompt": 12, "bitfit": 10, "kd": 10, "teacher": 10}
PATIENCE       = 3
HEAD_LR        = 5e-4
LR_DEFAULTS    = {"houlsby": 1e-3, "pfeiffer": 1e-3, "prefix": 2e-3, "prompt": 5e-3, "bitfit": 1e-3, "kd": 5e-5, "teacher": 3e-5}
WEIGHT_DECAY   = 0.01     # head / full-FT parameters. PEFT parameters use 0
WARMUP_FRAC    = 0.06
WEIGHT_CAP     = 15.0     # only when USE_CRF = False
USE_GC         = False    # set True on CUDA OOM
SEED           = 42
SMOKE          = False    # tiny dry run

CKPT_PATH = None
TOK_DIR   = None

# ----------------------------- Phase-1 architecture ----------------------------
D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS = 1024, 20, 16, 4
HEAD_DIM, D_FF, MAX_SEQ_LEN = D_MODEL // N_HEADS, 2432, 1024
ROPE_BASE, WINDOW_SIZE = 10000.0, 512

assert WIDTH + max(N_PROMPT, N_PREFIX) <= MAX_SEQ_LEN
OUT_ROOT = "/kaggle/working/lner_peft6"
os.makedirs(OUT_ROOT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "")
print("methods:", METHODS_TO_RUN, "| CRF", USE_CRF, "| BIDIR", BIDIRECTIONAL)


# unused here (only referenced by the first notebook's build_tagger kept in the reused cell)
METHOD, PARTIAL_LAYERS, RANK, ALPHA = 'full', 6, 8, 16.0


device: cuda | Tesla T4
methods: ['houlsby', 'pfeiffer', 'prefix', 'prompt', 'bitfit', 'kd'] | CRF True | BIDIR True


## Locate checkpoint & tokenizer

In [3]:
from transformers import PreTrainedTokenizerFast


def _find(patterns, prefer=()):
    hits = sorted({h for p in patterns for h in glob.glob(p, recursive=True)})
    for key in prefer:
        pref = [h for h in hits if key in h.lower()]
        if pref:
            return pref[0], hits
    return (hits[0] if hits else None), hits


if CKPT_PATH is None:
    CKPT_PATH, hits = _find(["/kaggle/input/**/best.pt"], prefer=("m4", "capstone"))
    print("best.pt candidates:", hits)
if TOK_DIR is None:
    tj, hits = _find(["/kaggle/input/**/tokenizer.json"], prefer=("legal_tokenizer", "tokenizer"))
    TOK_DIR = os.path.dirname(tj) if tj else None
    print("tokenizer.json candidates:", hits)
assert CKPT_PATH and os.path.isfile(CKPT_PATH), "best.pt not found - attach the checkpoint dataset or set CKPT_PATH"
assert TOK_DIR and os.path.isdir(TOK_DIR), "tokenizer not found - attach the tokenizer dataset or set TOK_DIR"
print("using checkpoint:", CKPT_PATH)
print("using tokenizer :", TOK_DIR)

tokenizer = PreTrainedTokenizerFast.from_pretrained(TOK_DIR)
tokenizer.model_max_length = 10 ** 9          # we window ourselves; silence the length warning
VOCAB_SIZE = len(tokenizer)
print(f"VOCAB_SIZE={VOCAB_SIZE} EOS={tokenizer.eos_token_id} PAD={tokenizer.pad_token_id} fast={tokenizer.is_fast}")
assert tokenizer.is_fast, "need a fast tokenizer for character offsets"

best.pt candidates: ['/kaggle/input/datasets/vivekchandwani7/m6-finalrun-ckpts/cs781_efficient/checkpoints/best.pt']
tokenizer.json candidates: ['/kaggle/input/datasets/paveetsaluja/m4-dataset-500m/legal_tokenizer_efficient/tokenizer.json']
using checkpoint: /kaggle/input/datasets/vivekchandwani7/m6-finalrun-ckpts/cs781_efficient/checkpoints/best.pt
using tokenizer : /kaggle/input/datasets/paveetsaluja/m4-dataset-500m/legal_tokenizer_efficient
VOCAB_SIZE=32000 EOS=1 PAD=0 fast=True


## Data

In [4]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P2 = "Phase_2_LNER"
LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
LABEL_SET = set(LABELS)
TAG_NAMES = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
K = len(TAG_NAMES)


def fetch(rel):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)


def read_json(path):
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)


def load_fold(text_rel, gold_rel=None):
    texts = read_json(fetch(text_rel))
    golds = read_json(fetch(gold_rel)) if gold_rel else {}
    rows, total, dropped, mismatch = [], 0, 0, 0
    for doc_id, text in texts.items():
        ents = []
        for it in golds.get(doc_id, []):
            s, e, lab = int(it[0]), int(it[1]), str(it[2])
            total += 1
            if lab in LABEL_SET and 0 <= s < e <= len(text):
                ents.append((s, e, lab))
                if len(it) > 3 and text[s:e] != it[3]:
                    mismatch += 1
            else:
                dropped += 1
        rows.append((str(doc_id), text, sorted(ents)))
    if gold_rel:
        print(f"{gold_rel}: {len(rows)} docs, {total} entities, dropped {dropped}, text[s:e] != entity_text for {mismatch}")
    return rows


fold1 = load_fold(f"{P2}/Train/fold1-text.json", f"{P2}/Train/fold1-gold.json")
fold2 = load_fold(f"{P2}/Train/fold2-text.json", f"{P2}/Train/fold2-gold.json")
test_rows = load_fold(f"{P2}/Test/fold3-text.json")
print("fold1", len(fold1), "fold2", len(fold2), "test", len(test_rows))

try:
    print("labels.txt:", open(fetch(f"{P2}/Train/labels.txt"), encoding="utf-8").read().split())
except Exception as exc:
    print("could not read labels.txt:", exc)
cnt = Counter(l for _, _, ents in fold1 + fold2 for _, _, l in ents)
print("entities per label:", {l: cnt[l] for l in LABELS})

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

Phase_2_LNER/Train/fold1-gold.json: 35 docs, 4120 entities, dropped 0, text[s:e] != entity_text for 0


fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

Phase_2_LNER/Train/fold2-gold.json: 35 docs, 4273 entities, dropped 0, text[s:e] != entity_text for 0


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

fold1 35 fold2 35 test 35


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

labels.txt: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
entities per label: {'APP': 407, 'RESP': 381, 'A.COUNSEL': 195, 'R.COUNSEL': 184, 'JUDGE': 245, 'WIT': 187, 'AUTH': 1003, 'COURT': 1013, 'STAT': 1260, 'PREC': 1183, 'DATE': 1575, 'CASENO': 760}


## Alignment / decoding / scoring utilities

In [5]:
def encode_doc(text):
    """Token ids + character offsets (end-exclusive) with surrounding whitespace removed from every token."""
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    offs = []
    for s, e in enc["offset_mapping"]:
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        offs.append((s, e))
    return enc["input_ids"], offs


def bio_tags(offs, ents):
    """Tag id per token. A token joins the entity it overlaps most (ties: first entity). Tag ids: 0=O, 1+2i=B-label_i, 2+2i=I-label_i.
    A zero-length (whitespace-only) token strictly inside an entity joins that entity so spans stay contiguous."""
    n = len(offs)
    starts = [o[0] for o in offs]
    ends = [o[1] for o in offs]
    owner = [-1] * n
    best = [0.0] * n
    for ei, (s, e, _lab) in enumerate(ents):
        i = bisect.bisect_right(ends, s)                    # first token whose end > s
        while i < n and starts[i] < e:
            if ends[i] > starts[i]:
                ov = min(ends[i], e) - max(starts[i], s)
            else:
                ov = 0.5 if (s < starts[i] < e) else 0.0
            if ov > best[i]:
                best[i], owner[i] = ov, ei
            i += 1
    tags = [0] * n
    for i, ei in enumerate(owner):
        if ei < 0:
            continue
        li = LABELS.index(ents[ei][2])
        tags[i] = 2 + 2 * li if (i > 0 and owner[i - 1] == ei) else 1 + 2 * li
    return tags


def window_ranges(n, width, stride):
    if n == 0:
        return []
    out, a = [], 0
    while True:
        b = min(a + width, n)
        out.append((a, b))
        if b == n:
            return out
        a += stride


def trim_span(text, s, e, mode):
    """mode 0: keep token boundaries. 1: drop trailing , ; : and spaces. 2: also drop a trailing '.' that is not an abbreviation dot."""
    if mode == 0:
        return s, e
    while e > s and (text[e - 1] in ",;:" or text[e - 1].isspace()):
        e -= 1
    while s < e and text[s].isspace():
        s += 1
    if mode == 2 and e - s >= 2 and text[e - 1] == "." and text[e - 2].isalnum():
        if not text[s:e].endswith((" v.", " vs.", " No.", " Ltd.", " Pvt.", " Dr.", " Mr.", " Ms.", " Mrs.", " Co.", " Inc.")):
            e -= 1
    return s, e


def decode_spans(text, offs, path, trim_mode=0):
    """BIO tag-id path -> unique (start, end, label) character spans (end exclusive), in document order."""
    spans, seen, i, n = [], set(), 0, len(path)
    while i < n:
        t = int(path[i])
        if t == 0:
            i += 1
            continue
        li = (t - 1) // 2
        j = i + 1
        while j < n and int(path[j]) == 2 + 2 * li:
            j += 1
        nz = [k for k in range(i, j) if offs[k][1] > offs[k][0]]
        if nz:
            s, e = trim_span(text, offs[nz[0]][0], offs[nz[-1]][1], trim_mode)
            sp = (s, e, LABELS[li])
            if e > s and sp not in seen:
                seen.add(sp)
                spans.append(sp)
        i = j
    return spans


def allowed_matrix(tag_names):
    k = len(tag_names)
    allowed = np.zeros((k, k), dtype=bool)
    for a, sa in enumerate(tag_names):
        for b, sb in enumerate(tag_names):
            if sb == "O" or sb.startswith("B-"):
                allowed[a, b] = True
            else:
                allowed[a, b] = sa in ("B-" + sb[2:], "I-" + sb[2:])
    return allowed


ALLOWED = allowed_matrix(TAG_NAMES)
HARD_TRANS = np.where(ALLOWED, 0.0, -1e4)          # transition scores used when there is no CRF


def log_softmax_np(x):
    x = x - x.max(axis=-1, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=-1, keepdims=True))


def viterbi(em, trans):
    """em: (T,K) emission scores, trans: (K,K) transition scores -> best tag-id path (list)."""
    T, k = em.shape
    if T == 0:
        return []
    score = em[0].copy()
    back = np.zeros((T, k), dtype=np.int32)
    cols = np.arange(k)
    for t in range(1, T):
        cand = score[:, None] + trans
        idx = cand.argmax(axis=0)
        back[t] = idx
        score = cand[idx, cols] + em[t]
    cur = int(score.argmax())
    path = [cur]
    for t in range(T - 1, 0, -1):
        cur = int(back[t, cur])
        path.append(cur)
    return path[::-1]


def best_path(logits, trans, use_crf, o_delta=0.0):
    em = logits.astype(np.float64) if use_crf else log_softmax_np(logits.astype(np.float64))
    em[:, 0] -= o_delta                       # o_delta > 0 trades precision for recall
    return viterbi(em, trans)


def strict_scores(gold_rows, preds):
    """Strict span+label macro P/R/F1 over the 12 labels (counts pooled over documents, as in the reference)."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc_id, _text, ents in gold_rows:
        g = set(ents)
        p = set(tuple(x[:3]) for x in preds.get(doc_id, []))
        for l in LABELS:
            gl = {x for x in g if x[2] == l}
            pl = {x for x in p if x[2] == l}
            tp[l] += len(gl & pl)
            fp[l] += len(pl - gl)
            fn[l] += len(gl - pl)
    per, ps, rs, fs = {}, [], [], []
    for l in LABELS:
        pr = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        rc = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        per[l] = (pr, rc, f1, tp[l], fp[l], fn[l])
        ps.append(pr); rs.append(rc); fs.append(f1)
    return {"f1": float(np.mean(fs)), "p": float(np.mean(ps)), "r": float(np.mean(rs)), "per": per}


def print_scores(sc):
    for l in LABELS:
        pr, rc, f1, tp_, fp_, fn_ = sc["per"][l]
        print(f"  {l:10s} P={pr:.3f} R={rc:.3f} F1={f1:.3f}  (tp {tp_}, fp {fp_}, fn {fn_})")
    print(f"  MACRO      P={sc['p']:.3f} R={sc['r']:.3f} F1={sc['f1']:.3f}")

## Offset pipeline check

In [6]:
PREP = {}                                  # doc_id -> (ids, offsets)


def prepare(rows):
    for doc_id, text, _ in tqdm(rows, desc="tokenizing", leave=False):
        if doc_id not in PREP:
            PREP[doc_id] = encode_doc(text)


all_train = fold1 + fold2
prepare(all_train)
lens = np.array([len(PREP[d][0]) for d, _, _ in all_train])
print(f"tokens/doc: mean {lens.mean():.0f}, median {np.median(lens):.0f}, max {lens.max()}; docs longer than WIDTH: {(lens > WIDTH).mean():.0%}")

# whitespace-trimmed offsets must be ordered, inside the text, and never start/end on whitespace
for doc_id, text, _ in all_train[:20]:
    ids, offs = PREP[doc_id]
    assert all(0 <= s <= e <= len(text) for s, e in offs), "offset outside text"
    assert all(a[0] <= b[0] and a[1] <= b[1] for a, b in zip(offs, offs[1:])), "offsets not monotonic"

covered = total = shown = 0
for doc_id, text, ents in all_train:
    ids, offs = PREP[doc_id]
    st = {s for s, e in offs if e > s}
    en = {e for s, e in offs if e > s}
    for s, e, l in ents:
        total += 1
        if s in st and e in en:
            covered += 1
        elif shown < 8:
            shown += 1
            print(f"  off-boundary {l}: {text[s:e][:50]!r} in ...{text[max(0, s - 10):e + 10][:80]!r}")
print(f"gold spans on token boundaries: {covered}/{total} = {covered / max(total, 1):.3f}")

oracle = {}
for mode in (0, 1, 2):
    preds = {d: decode_spans(t, PREP[d][1], bio_tags(PREP[d][1], ents), mode) for d, t, ents in all_train}
    oracle[mode] = strict_scores(all_train, preds)
    print(f"oracle round-trip, trim mode {mode}: macro-F1 {oracle[mode]['f1']:.4f}")
TRIM_MODE = max((0, 1, 2), key=lambda m: (oracle[m]["f1"], -m))
print("TRIM_MODE =", TRIM_MODE, "(oracle per label below)")
print_scores(oracle[TRIM_MODE])

tokenizing:   0%|          | 0/70 [00:00<?, ?it/s]

tokens/doc: mean 6550, median 5538, max 23814; docs longer than WIDTH: 100%
  off-boundary PREC: 'State of M.P. vs. Kashiram & Ors.' in ...'e case of State of M.P. vs. Kashiram & Ors.[1], the s'
  off-boundary PREC: 'Gambangi Appalaswamy Naldu and Others v. Behara Ve' in ...'1565] and Gambangi Appalaswamy Naldu and Others v. Behara Venkataramanayya Patro'
  off-boundary AUTH: 'Railway Board' in ...' down the Railway Boards instruct'
  off-boundary AUTH: 'Railway Board' in ...'tation of Railway Boards decision'
  off-boundary PREC: 'Shankarsan Dash Vs. Union of India [1991 (3) SCC 4' in ...' cases in Shankarsan Dash Vs. Union of India [1991 (3) SCC 47]. Para 7 o'
  off-boundary PREC: 'Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 1' in ...'C 268] or Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]. Hence th'
  off-boundary STAT: 'Constitution' in ...'After the Constitutional Amendme'
  off-boundary AUTH: 'State Government' in ...'uding the State Governments and the '
gold spans 

## Phase-1 model + checkpoint

In [7]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).type_as(x) * self.weight


def precompute_rope_freqs(dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, freqs_cos, freqs_sin):
    x_r = x.reshape(*x.shape[:-1], -1, 2)
    x0, x1 = x_r[..., 0], x_r[..., 1]
    seq_len = x.shape[2]
    cos = freqs_cos[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    sin = freqs_sin[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    out0 = x0 * cos - x1 * sin
    out1 = x0 * sin + x1 * cos
    return torch.stack([out0, out1], dim=-1).reshape_as(x)


class SwiGLU_FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    B, H, T, D = x.shape
    return x[:, :, None, :, :].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.n_heads = n_heads; self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim; self.n_rep = n_heads // n_kv_heads
        self.use_window = use_window; self.window_size = window_size
        self.attn_p = dropout
        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, freqs_cos, freqs_sin)
        k = apply_rope(k, freqs_cos, freqs_sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v) if use_cache else None
        dp = self.attn_p if self.training else 0.0
        if past_kv is not None:
            if self.use_window:
                k = k[:, :, -self.window_size:]; v = v[:, :, -self.window_size:]
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp)
        else:
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            if self.use_window and T > 1:
                row = torch.arange(T, device=x.device).unsqueeze(1)
                col = torch.arange(T, device=x.device).unsqueeze(0)
                mask = (col <= row) & (col > row - self.window_size)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp)
            else:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dp)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.out_proj(out)), new_kv


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAttention(d_model, n_heads, n_kv_heads, head_dim, dropout, use_window, window_size)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU_FFN(d_model, d_ff, dropout)

    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        attn_out, new_kv = self.attn(self.attn_norm(x), freqs_cos, freqs_sin, use_cache, past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv


class EfficientLegalLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, head_dim, d_ff,
                 max_seq_len, rope_base=10000.0, dropout=0.0, window_size=256, use_gc=True):
        super().__init__()
        self.d_model = d_model; self.n_layers = n_layers; self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len; self.use_gc = use_gc
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout,
                             use_window=(i % 2 == 0), window_size=window_size)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        fc, fs = precompute_rope_freqs(head_dim, max_seq_len, rope_base)
        self.register_buffer("freqs_cos", fc, persistent=False)
        self.register_buffer("freqs_sin", fs, persistent=False)

    def forward(self, input_ids, targets=None, use_cache=False, past_kvs=None):
        B, T = input_ids.shape
        x = self.emb_dropout(self.tok_emb(input_ids))
        past_len = past_kvs[0][0].shape[2] if past_kvs is not None else 0
        fc = self.freqs_cos[past_len:past_len + T]
        fs = self.freqs_sin[past_len:past_len + T]
        presents = [] if use_cache else None
        for i, layer in enumerate(self.layers):
            pkv = past_kvs[i] if past_kvs is not None else None
            x, kv = layer(x, fc, fs, use_cache, pkv)
            if use_cache:
                presents.append(kv)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits[:, :-1, :].contiguous().view(-1, self.vocab_size),
                                   targets[:, 1:].contiguous().view(-1), ignore_index=-100)
        return logits, loss, presents


base = EfficientLegalLM(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF,
                        MAX_SEQ_LEN, ROPE_BASE, 0.0, WINDOW_SIZE, use_gc=False)
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state = ckpt["model_state_dict"]
# capstone-m4 saved the torch.compile()d model, so keys carry an "_orig_mod." prefix
state = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in state.items()}
missing, unexpected = base.load_state_dict(state, strict=False)
missing = [k for k in missing if k != "lm_head.weight"]      # tied to tok_emb.weight, so a copy is not required
assert not missing and not unexpected, f"checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}"
print(f"loaded {CKPT_PATH}  (pre-training step {ckpt.get('step')}, loss {ckpt.get('loss')})")
del ckpt, state
base = base.to(dev).eval()

# sanity: a correctly loaded LM has a low next-token loss on legal text (random weights give ~ln(32000) = 10.4)
ids0 = torch.tensor([PREP[all_train[0][0]][0][:512]], device=dev)
with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
    lm_loss = base(ids0, targets=ids0)[1].item()
print(f"LM sanity loss on a training judgment: {lm_loss:.3f}  (expect roughly 2-4)")
assert lm_loss < 6.0, "LM loss is too high - weights or tokenizer do not match"

loaded /kaggle/input/datasets/vivekchandwani7/m6-finalrun-ckpts/cs781_efficient/checkpoints/best.pt  (pre-training step 7629, loss 2.381956102848053)
LM sanity loss on a training judgment: 2.307  (expect roughly 2-4)


## Tagger base (masks, CRF, LegalTagger) — unchanged

In [8]:
def build_masks(T, pad_mask, bidirectional, window):
    """Boolean attention masks (True = may attend), shape (B,1,T,T): one for window layers, one for full layers."""
    r = torch.arange(T, device=pad_mask.device).unsqueeze(1)
    c = torch.arange(T, device=pad_mask.device).unsqueeze(0)
    if bidirectional:
        full = torch.ones(T, T, dtype=torch.bool, device=pad_mask.device)
        win = (c - r).abs() < window
    else:
        full = c <= r
        win = (c <= r) & (c > r - window)
    key = pad_mask[:, None, None, :]
    return full[None, None] & key, win[None, None] & key


def block_forward(blk, x, cos, sin, mask):
    a = blk.attn
    h = blk.attn_norm(x)
    B, T, _ = h.shape
    q = a.q_proj(h).view(B, T, a.n_heads, a.head_dim).transpose(1, 2)
    k = a.k_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    v = a.v_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)
    k = repeat_kv(k, a.n_rep)
    v = repeat_kv(v, a.n_rep)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, T, -1)
    x = x + a.out_proj(out)
    return x + blk.ffn(blk.ffn_norm(x))


class LoRALinear(nn.Module):
    """y = W x + (alpha/r) B A x   (base W frozen, B starts at zero)."""
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f, in_f = base_linear.weight.shape
        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank

    def forward(self, x):
        return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())


class LoRMALinear(nn.Module):
    """LoRMA (multiplicative): y = (I + (alpha/r) B A) W x   (base W frozen, B starts at zero -> identical to Phase 1 at step 0)."""
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f = base_linear.weight.shape[0]
        self.A = nn.Parameter(torch.randn(rank, out_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * ((h @ self.A.t()) @ self.B.t())


def adapt_linears(backbone, kind, rank, alpha):
    wrapper = LoRMALinear if kind == "lorma" else LoRALinear
    n = 0
    for blk in backbone.layers:
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if isinstance(child, nn.Linear):
                    setattr(parent, name, wrapper(child, rank, alpha))
                    n += 1
    return n


class ChainCRF(nn.Module):
    """Linear-chain CRF over BIO tags; illegal transitions (O->I-x, B-x->I-y, ...) are blocked."""
    def __init__(self, tag_names):
        super().__init__()
        self.transitions = nn.Parameter(torch.zeros(len(tag_names), len(tag_names)))
        self.register_buffer("allowed", torch.from_numpy(allowed_matrix(tag_names)))

    def transition(self):
        return self.transitions.masked_fill(~self.allowed, -1e4)

    def nll(self, em, tags, mask):
        trans = self.transition()
        safe = tags.clamp(min=0)
        mf = mask.float()
        emit = em.gather(2, safe.unsqueeze(2)).squeeze(2)
        gold = (emit * mf).sum(1) + (trans[safe[:, :-1], safe[:, 1:]] * mf[:, 1:]).sum(1)
        alpha = em[:, 0]
        for t in range(1, em.size(1)):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + trans.unsqueeze(0) + em[:, t].unsqueeze(1), dim=1)
            alpha = torch.where(mask[:, t].unsqueeze(1), nxt, alpha)
        logz = torch.logsumexp(alpha, dim=-1)
        return ((logz - gold) / mf.sum(1).clamp(min=1)).mean()


class LegalTagger(nn.Module):
    def __init__(self, backbone, tag_names, use_crf, bidirectional, bidir_window, use_gc):
        super().__init__()
        self.backbone = backbone
        self.bidirectional, self.bidir_window, self.use_gc = bidirectional, bidir_window, use_gc
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(D_MODEL, len(tag_names))
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
        with torch.no_grad():
            self.head.bias[0] = 2.0                      # start by predicting "O"
        self.crf = ChainCRF(tag_names) if use_crf else None

    def encode(self, ids, pad_mask):
        m = self.backbone
        T = ids.size(1)
        x = m.tok_emb(ids)
        cos, sin = m.freqs_cos[:T], m.freqs_sin[:T]
        window = self.bidir_window if self.bidirectional else WINDOW_SIZE
        full, win = build_masks(T, pad_mask, self.bidirectional, window)
        for blk in m.layers:
            mask = win if blk.attn.use_window else full
            if self.use_gc and self.training:
                x = grad_checkpoint(block_forward, blk, x, cos, sin, mask, use_reentrant=False)
            else:
                x = block_forward(blk, x, cos, sin, mask)
        return m.final_norm(x)

    def forward(self, ids, pad_mask):
        return self.head(self.drop(self.encode(ids, pad_mask)))


def build_tagger():
    bb = base
    for p in bb.parameters():
        p.requires_grad_(METHOD == "full")
    if METHOD == "partial":
        for blk in bb.layers[-PARTIAL_LAYERS:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in bb.final_norm.parameters():
            p.requires_grad_(True)
    elif METHOD in ("lora", "lorma"):
        print(f"wrapped {adapt_linears(bb, METHOD, RANK, ALPHA)} linear layers with {METHOD}")
    tg = LegalTagger(bb, TAG_NAMES, USE_CRF, BIDIRECTIONAL, BIDIR_WINDOW, USE_GC).to(dev)
    tr = sum(p.numel() for p in tg.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in tg.parameters())
    print(f"parameters: trainable {tr / 1e6:.2f}M / total {tot / 1e6:.2f}M ({tr / tot:.1%})")
    return tg

## Windows, dev split, collate

In [9]:
fold2_sorted = sorted(fold2, key=lambda r: r[0])
dev_rows = fold2_sorted[-DEV_COUNT:]
dev_ids = {r[0] for r in dev_rows}
train_rows = list(fold1) + [r for r in fold2 if r[0] not in dev_ids]
if SMOKE:
    train_rows, dev_rows = train_rows[:4], dev_rows[:2]
print("train docs", len(train_rows), "| dev docs", [r[0] for r in dev_rows])
prepare(train_rows + dev_rows + test_rows)

train_windows = []                                           # (token ids, tag ids)
for doc_id, text, ents in train_rows:
    ids, offs = PREP[doc_id]
    tags = bio_tags(offs, ents)
    for a, b in window_ranges(len(ids), WIDTH, STRIDE):
        train_windows.append((ids[a:b], tags[a:b]))
print(f"train windows: {len(train_windows)} (width {WIDTH}, stride {STRIDE})")

tag_counts = np.ones(K)
for _ids, tgs in train_windows:
    for t in tgs:
        tag_counts[t] += 1
w = 1.0 / np.sqrt(tag_counts)
w = w * (K / w.sum())
w = np.minimum(w, np.median(w) * WEIGHT_CAP)
cls_weights = torch.tensor(w, dtype=torch.float32, device=dev)   # only used when USE_CRF = False
use_amp = dev.type == "cuda"


def collate(items):
    L = max(len(i) for i, _ in items)
    ids = torch.zeros(len(items), L, dtype=torch.long)
    tgt = torch.full((len(items), L), -100, dtype=torch.long)
    pm = torch.zeros(len(items), L, dtype=torch.bool)
    for r, (i, t) in enumerate(items):
        ids[r, :len(i)] = torch.tensor(i)
        tgt[r, :len(t)] = torch.tensor(t)
        pm[r, :len(i)] = True
    return ids.to(dev), tgt.to(dev), pm.to(dev)


train docs 62 | dev docs ['756812', '76640285', '79816183', '806212', '83742049', '87753577', '90251163', '952082']


tokenizing:   0%|          | 0/105 [00:00<?, ?it/s]

train windows: 1541 (width 512, stride 256)


## Inference helpers — unchanged

In [10]:
@torch.no_grad()
def doc_logits(tg, ids):
    """Per-token tag logits for a whole document: overlapping windows, centre-weighted average. -> (n_tokens, K) float32."""
    tg.eval()
    n = len(ids)
    ids_t = torch.tensor(ids, dtype=torch.long)
    acc = torch.zeros(n, K, device=dev)
    wsum = torch.zeros(n, 1, device=dev)
    ranges = window_ranges(n, WIDTH, STRIDE)
    for i in range(0, len(ranges), INFER_BATCH):
        chunk = ranges[i:i + INFER_BATCH]
        L = max(b - a for a, b in chunk)
        x = torch.zeros(len(chunk), L, dtype=torch.long)
        pm = torch.zeros(len(chunk), L, dtype=torch.bool)
        for r, (a, b) in enumerate(chunk):
            x[r, :b - a] = ids_t[a:b]
            pm[r, :b - a] = True
        x, pm = x.to(dev), pm.to(dev)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            out = tg(x, pm)
        out = out.float()
        for r, (a, b) in enumerate(chunk):
            m = b - a
            wt = torch.minimum(torch.arange(1, m + 1, device=dev), torch.arange(m, 0, -1, device=dev)).float().unsqueeze(1)
            acc[a:b] += out[r, :m] * wt
            wsum[a:b] += wt
    return (acc / wsum.clamp(min=1e-6)).cpu().numpy()


def get_trans(tg):
    if tg.crf is not None:
        return tg.crf.transition().detach().float().cpu().numpy().astype(np.float64)
    return HARD_TRANS


def build_cache(tg, rows, desc="scoring"):
    return [(d, t, PREP[d][1], doc_logits(tg, PREP[d][0])) for d, t, _ in tqdm(rows, desc=desc, leave=False)]


def decode_cache(cache, trans, o_delta=0.0, trim_mode=0):
    return {d: decode_spans(t, offs, best_path(lg, trans, USE_CRF, o_delta), trim_mode) for d, t, offs, lg in cache}


def evaluate_dev(tg):
    cache = build_cache(tg, dev_rows)
    return strict_scores(dev_rows, decode_cache(cache, get_trans(tg), 0.0, TRIM_MODE))

## New methods: adapters, prefix, prompt, BitFit, KD student
* **Houlsby**: bottleneck adapter after attention *and* after FFN in every block.
* **Pfeiffer**: single (wider) adapter after the FFN only.
* **Prefix**: learnable per-layer K/V prefix, initialised from real activations.
* **Prompt**: soft prompt embeddings prepended to the input (initialised from real token embeddings, always visible to all tokens).
* **BitFit**: the base model has no biases, so zero-init biases are added and trained together with the RMSNorm gains.
* **KD**: full-FT teacher → pruned 10-layer student (hard CRF loss + temperature-scaled KL on emissions).

In [11]:
import copy, gc

class Adapter(nn.Module):
    """Bottleneck adapter: h + W_up(gelu(W_down h)). W_up is zero-initialised, so it starts as the identity."""
    def __init__(self, d, r):
        super().__init__()
        self.down = nn.Linear(d, r)
        self.up = nn.Linear(r, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class BiasLinear(nn.Module):
    """Frozen base weight (shared) + NEW trainable zero-initialised bias. The Phase-1 linears have no bias, BitFit needs one."""
    def __init__(self, lin):
        super().__init__()
        self.weight = lin.weight
        self.bias = nn.Parameter(torch.zeros(lin.weight.shape[0], device=lin.weight.device))

    def forward(self, x):
        return F.linear(x, self.weight, self.bias)


def add_biases(backbone):
    n = 0
    for blk in backbone.layers:
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if isinstance(child, nn.Linear):
                    setattr(parent, name, BiasLinear(child)); n += 1
    return n


def block_forward_x(blk, x, cos, sin, mask, ad_attn=None, ad_ffn=None, pk=None, pv=None):
    """block_forward + optional adapters (after attention / after FFN) + optional learnable K/V prefix."""
    a = blk.attn
    h = blk.attn_norm(x)
    B, T, _ = h.shape
    q = a.q_proj(h).view(B, T, a.n_heads, a.head_dim).transpose(1, 2)
    k = a.k_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    v = a.v_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)
    if pk is not None:                                   # prefixes carry no position (they are not rotated)
        k = torch.cat([pk.unsqueeze(0).expand(B, -1, -1, -1).to(k.dtype), k], dim=2)
        v = torch.cat([pv.unsqueeze(0).expand(B, -1, -1, -1).to(v.dtype), v], dim=2)
    k = repeat_kv(k, a.n_rep)
    v = repeat_kv(v, a.n_rep)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, T, -1)
    ao = a.out_proj(out)
    if ad_attn is not None:
        ao = ad_attn(ao)
    x = x + ao
    fo = blk.ffn(blk.ffn_norm(x))
    if ad_ffn is not None:
        fo = ad_ffn(fo)
    return x + fo


class LegalTaggerX(LegalTagger):
    """LegalTagger + houlsby / pfeiffer adapters, prefix tuning, prompt tuning. (bitfit / kd / full only change which params train)."""
    def __init__(self, backbone, method, tag_names, use_crf, bidirectional, bidir_window, use_gc):
        super().__init__(backbone, tag_names, use_crf, bidirectional, bidir_window, use_gc)
        self.method = method
        L = len(backbone.layers)
        self.n_prompt = N_PROMPT if method == "prompt" else 0
        self.n_prefix = N_PREFIX if method == "prefix" else 0
        self.ad_attn = nn.ModuleList([Adapter(D_MODEL, ADAPTER_R[method]) for _ in range(L)]) if method == "houlsby" else None
        self.ad_ffn = nn.ModuleList([Adapter(D_MODEL, ADAPTER_R[method]) for _ in range(L)]) if method in ("houlsby", "pfeiffer") else None
        if method == "prompt":
            self.prompt = nn.Parameter(torch.randn(N_PROMPT, D_MODEL) * 0.02)
        if method == "prefix":
            self.pk = nn.ParameterList([nn.Parameter(torch.randn(N_KV_HEADS, N_PREFIX, HEAD_DIM) * 0.1) for _ in range(L)])
            self.pv = nn.ParameterList([nn.Parameter(torch.randn(N_KV_HEADS, N_PREFIX, HEAD_DIM) * 0.1) for _ in range(L)])

    @torch.no_grad()
    def init_soft(self, sample_ids):
        """prompt: copy embeddings of real tokens.  prefix: copy the real (post-RoPE) keys/values of the first tokens of a document."""
        m = self.backbone
        if self.method == "prompt":
            pick = torch.tensor(random.sample(sample_ids, self.n_prompt), device=dev)
            self.prompt.data.copy_(m.tok_emb(pick).float())
        elif self.method == "prefix":
            P = self.n_prefix
            x = m.tok_emb(torch.tensor([sample_ids[:P]], device=dev))
            cos, sin = m.freqs_cos[:P], m.freqs_sin[:P]
            for i, blk in enumerate(m.layers):
                a = blk.attn
                h = blk.attn_norm(x)
                k = apply_rope(a.k_proj(h).view(1, P, a.n_kv_heads, a.head_dim).transpose(1, 2), cos, sin)
                v = a.v_proj(h).view(1, P, a.n_kv_heads, a.head_dim).transpose(1, 2)
                self.pk[i].data.copy_(k[0].float()); self.pv[i].data.copy_(v[0].float())
                x = block_forward(blk, x, cos, sin, None)

    def encode(self, ids, pad_mask):
        m = self.backbone
        B = ids.size(0)
        x = m.tok_emb(ids)
        P = self.n_prompt
        if P:
            x = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1).to(x.dtype), x], dim=1)
            pad_mask = torch.cat([torch.ones(B, P, dtype=torch.bool, device=ids.device), pad_mask], dim=1)
        T = x.size(1)
        cos, sin = m.freqs_cos[:T], m.freqs_sin[:T]
        window = self.bidir_window if self.bidirectional else WINDOW_SIZE
        full, win = build_masks(T, pad_mask, self.bidirectional, window)
        if P:                                            # every token may always read the soft prompt, also in the sliding-window layers
            win = win | (torch.arange(T, device=ids.device) < P)[None, None, None, :]
        if self.n_prefix:
            full = F.pad(full, (self.n_prefix, 0), value=True)
            win = F.pad(win, (self.n_prefix, 0), value=True)
        for i, blk in enumerate(m.layers):
            mask = win if blk.attn.use_window else full
            aa = self.ad_attn[i] if self.ad_attn is not None else None
            af = self.ad_ffn[i] if self.ad_ffn is not None else None
            pk = self.pk[i] if self.n_prefix else None
            pv = self.pv[i] if self.n_prefix else None
            if self.use_gc and self.training:
                x = grad_checkpoint(block_forward_x, blk, x, cos, sin, mask, aa, af, pk, pv, use_reentrant=False)
            else:
                x = block_forward_x(blk, x, cos, sin, mask, aa, af, pk, pv)
        return m.final_norm(x)[:, P:]                    # strip the soft prompt -> same shape as the plain tagger


def fresh_base():
    """A new, untouched Phase-1 backbone (so no method inherits another method's modifications)."""
    b = EfficientLegalLM(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF,
                         MAX_SEQ_LEN, ROPE_BASE, 0.0, WINDOW_SIZE, use_gc=False)
    ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
    st = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in ck["model_state_dict"].items()}
    b.load_state_dict(st, strict=False)
    del ck, st
    gc.collect()
    return b.to(dev).eval()


def build_tagger_x(method):
    """method in houlsby | pfeiffer | prefix | prompt | bitfit | full (teacher)."""
    bb = fresh_base()
    for p in bb.parameters():
        p.requires_grad_(method == "full")
    if method == "bitfit":
        print(f"added zero-init biases to {add_biases(bb)} linear layers")
        for blk in bb.layers:
            for mod in blk.modules():
                if isinstance(mod, BiasLinear):
                    mod.bias.requires_grad_(True)
            blk.attn_norm.weight.requires_grad_(True); blk.ffn_norm.weight.requires_grad_(True)
        bb.final_norm.weight.requires_grad_(True)
    tg = LegalTaggerX(bb, method, TAG_NAMES, USE_CRF, BIDIRECTIONAL, BIDIR_WINDOW, USE_GC).to(dev)
    if method in ("prompt", "prefix"):
        tg.init_soft(list(PREP[train_rows[0][0]][0]))
    return tg


def build_student(teacher):
    """Pruned student: deep copy of the teacher, keeping only the layers in KD_KEEP (head + CRF copied too)."""
    bb = copy.deepcopy(teacher.backbone)
    bb.layers = nn.ModuleList([bb.layers[i] for i in KD_KEEP])
    bb.n_layers = len(KD_KEEP)
    for p in bb.parameters():
        p.requires_grad_(True)
    st = LegalTaggerX(bb, "kd", TAG_NAMES, USE_CRF, BIDIRECTIONAL, BIDIR_WINDOW, USE_GC).to(dev)
    st.head.load_state_dict(teacher.head.state_dict())
    if st.crf is not None:
        st.crf.load_state_dict(teacher.crf.state_dict())
    return st


## Training / tuning / submission helpers

In [12]:
def hard_loss(tg, em, tgt, pm):
    if tg.crf is not None:
        return tg.crf.nll(em, tgt, pm)
    return F.cross_entropy(em.reshape(-1, K), tgt.reshape(-1), weight=cls_weights, ignore_index=-100)


def kd_loss(s_em, t_em, pm, T=KD_TEMP):
    """Temperature-scaled KL(teacher || student) over the tag emissions of the valid (non-pad) tokens."""
    p = F.softmax(t_em / T, dim=-1)
    lq = F.log_softmax(s_em / T, dim=-1)
    kl = (p * (torch.log(p.clamp(min=1e-9)) - lq)).sum(-1)
    return (kl * pm.float()).sum() / pm.float().sum().clamp(min=1) * T * T


def make_optimizer(tg, lr, full_ft):
    head = [p for n, p in tg.named_parameters() if p.requires_grad and (n.startswith("head.") or n.startswith("crf."))]
    rest = [p for n, p in tg.named_parameters() if p.requires_grad and not (n.startswith("head.") or n.startswith("crf."))]
    groups = [{"params": head, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY}]
    if rest:
        groups.append({"params": rest, "lr": lr, "weight_decay": WEIGHT_DECAY if full_ft else 0.0})
    return torch.optim.AdamW(groups)


def fit(tg, tag, teacher=None):
    """Train with early stopping on dev strict macro-F1. Returns (best dev F1, checkpoint path)."""
    lr, epochs = LR_DEFAULTS[tag], (1 if SMOKE else EPOCHS_BY[tag])
    full_ft = tag in ("teacher", "kd")
    tr = sum(p.numel() for p in tg.parameters() if p.requires_grad); tot = sum(p.numel() for p in tg.parameters())
    print(f"[{tag}] trainable {tr / 1e6:.2f}M / total {tot / 1e6:.2f}M ({tr / tot:.2%}) | lr {lr} | epochs {epochs}")
    trainable = [p for p in tg.parameters() if p.requires_grad]
    opt = make_optimizer(tg, lr, full_ft)
    spe = math.ceil(len(train_windows) / BATCH); total = spe * epochs; warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.05, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    ckpt_path = os.path.join(OUT_ROOT, f"best_{tag}.pt")
    best, bad, t0 = -1.0, 0, time.time()
    for ep in range(1, epochs + 1):
        tg.train()
        order = np.random.permutation(len(train_windows)); tl, nb = 0.0, 0
        bar = tqdm(range(0, len(order), BATCH), desc=f"{tag} ep{ep}/{epochs}", leave=False)
        for bi in bar:
            ids, tgt, pm = collate([train_windows[i] for i in order[bi:bi + BATCH]])
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                em = tg(ids, pm)
                if teacher is not None:
                    with torch.no_grad():
                        t_em = teacher(ids, pm)
            em = em.float()
            loss = hard_loss(tg, em, tgt, pm)
            if teacher is not None:
                loss = (1 - KD_ALPHA) * loss + KD_ALPHA * kd_loss(em, t_em.float(), pm)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss.detach()); nb += 1
            bar.set_postfix(loss=f"{tl / nb:.4f}")
        sc = evaluate_dev(tg)
        print(f"  ep {ep}: loss {tl / nb:.4f} | dev strict P {sc['p']:.4f} R {sc['r']:.4f} F1 {sc['f1']:.4f} | {(time.time() - t0) / 60:.1f} min")
        if sc["f1"] > best:
            best, bad = sc["f1"], 0
            torch.save({"model": tg.state_dict(), "dev_f1": best, "epoch": ep}, ckpt_path)
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  early stop"); break
    del opt, scaler
    return best, ckpt_path


DELTAS = [0.0, -0.5, 0.5, 1.0, 1.5, 2.0, 3.0]
MODES = [TRIM_MODE] + [m for m in (0, 1, 2) if m != TRIM_MODE]


def tune_decode(dev_cache, trans):
    best_cfg, best_sc = (0.0, TRIM_MODE), None
    for delta in DELTAS:
        paths = [(d, t, offs, best_path(lg, trans, USE_CRF, delta)) for d, t, offs, lg in dev_cache]
        for mode in MODES:
            sc = strict_scores(dev_rows, {d: decode_spans(t, offs, p, mode) for d, t, offs, p in paths})
            if best_sc is None or sc["f1"] > best_sc["f1"] + 1e-9:
                best_sc, best_cfg = sc, (delta, mode)
    return best_cfg, best_sc


def write_submission(test_preds, tag):
    payload = {d: [[int(s), int(e), lab] for s, e, lab in test_preds[d]] for d, _, _ in test_rows}
    pred = os.path.join(OUT_ROOT, f"prediction_{tag}.json")
    with open(pred, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False)
    zp = f"/kaggle/working/submission_{tag}.zip"
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pred, arcname="prediction.json")          # prediction.json at the zip root
    return zp


RESULTS, CACHES = {}, {}


def finish(tg, tag, ckpt_path, minutes):
    """Reload the best epoch, tune (o_delta, trim) on dev, predict test, write submission_{tag}.zip, remember logits for the ensemble."""
    tg.load_state_dict(torch.load(ckpt_path, map_location=dev)["model"])
    trans = get_trans(tg)
    dev_cache = build_cache(tg, dev_rows, f"dev logits ({tag})")
    (delta, mode), sc = tune_decode(dev_cache, trans)
    print(f"[{tag}] o_delta {delta}, trim {mode} | dev strict P {sc['p']:.4f} R {sc['r']:.4f} F1 {sc['f1']:.4f}")
    print_scores(sc)
    test_cache = build_cache(tg, test_rows, f"test logits ({tag})")
    zp = write_submission(decode_cache(test_cache, trans, delta, mode), tag)
    print("saved", zp)
    CACHES[tag] = (dev_cache, test_cache, trans)
    RESULTS[tag] = {"P": sc["p"], "R": sc["r"], "F1": sc["f1"], "delta": delta, "trim": mode,
                    "trainable_M": sum(p.numel() for p in tg.parameters() if p.requires_grad) / 1e6, "min": minutes}


## Run all methods

In [13]:
def run_method(tag):
    print("\n" + "=" * 80 + f"\nSTART {tag.upper()}\n" + "=" * 80)
    t0 = time.time()
    if tag == "kd":
        teacher = build_tagger_x("full")
        if TEACHER_CKPT and os.path.exists(TEACHER_CKPT):
            teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=dev)["model"]); print("teacher loaded from", TEACHER_CKPT)
        else:
            tpath = os.path.join(OUT_ROOT, "best_teacher.pt")
            if os.path.exists(tpath):
                teacher.load_state_dict(torch.load(tpath, map_location=dev)["model"]); print("teacher reused from", tpath)
            else:
                _, tpath = fit(teacher, "teacher")
                teacher.load_state_dict(torch.load(tpath, map_location=dev)["model"])
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        print(f"teacher dev strict F1: {evaluate_dev(teacher)['f1']:.4f}")
        student = build_student(teacher)
        _, path = fit(student, "kd", teacher=teacher)
        finish(student, "kd", path, (time.time() - t0) / 60)
        del teacher, student
    else:
        tg = build_tagger_x(tag)
        _, path = fit(tg, tag)
        finish(tg, tag, path, (time.time() - t0) / 60)
        del tg
    gc.collect(); torch.cuda.empty_cache()


for _tag in METHODS_TO_RUN:
    run_method(_tag)



START HOULSBY
[houlsby] trainable 2.69M / total 237.35M (1.13%) | lr 0.001 | epochs 10


houlsby ep1/10:   0%|          | 0/193 [00:00<?, ?it/s]

/tmp/ipykernel_22/3480639942.py:55: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


scoring:   0%|          | 0/8 [00:00<?, ?it/s]

  ep 1: loss 0.2485 | dev strict P 0.4864 R 0.3435 F1 0.3908 | 2.1 min


houlsby ep2/10:   0%|          | 0/193 [00:00<?, ?it/s]

... (189 outputs left out)

dev logits (kd):   0%|          | 0/8 [00:00<?, ?it/s]

[kd] o_delta 1.0, trim 1 | dev strict P 0.5327 R 0.5610 F1 0.5312
  APP        P=0.192 R=0.061 F1=0.093  (tp 5, fp 21, fn 77)
  RESP       P=0.152 R=0.059 F1=0.085  (tp 5, fp 28, fn 80)
  A.COUNSEL  P=0.733 R=0.524 F1=0.611  (tp 11, fp 4, fn 10)
  R.COUNSEL  P=0.450 R=0.750 F1=0.563  (tp 9, fp 11, fn 3)
  JUDGE      P=0.789 R=0.714 F1=0.750  (tp 15, fp 4, fn 6)
  WIT        P=0.241 R=0.500 F1=0.326  (tp 14, fp 44, fn 14)
  AUTH       P=0.486 R=0.438 F1=0.461  (tp 53, fp 56, fn 68)
  COURT      P=0.867 R=0.917 F1=0.892  (tp 111, fp 17, fn 10)
  STAT       P=0.866 R=0.792 F1=0.827  (tp 213, fp 33, fn 56)
  PREC       P=0.378 R=0.567 F1=0.453  (tp 51, fp 84, fn 39)
  DATE       P=0.909 R=0.966 F1=0.937  (tp 170, fp 17, fn 6)
  CASENO     P=0.329 R=0.444 F1=0.378  (tp 24, fp 49, fn 30)
  MACRO      P=0.533 R=0.561 F1=0.531


test logits (kd):   0%|          | 0/35 [00:00<?, ?it/s]

saved /kaggle/working/submission_kd.zip


## Ensemble + summary

In [14]:
# Logit-average ensemble over every method that finished in the loop above (cheap, usually the biggest single metric gain).
if RUN_ENSEMBLE and len(CACHES) >= 2:
    tags = list(CACHES)
    dev_ens = [(d, t, offs, np.mean([CACHES[g][0][i][3] for g in tags], axis=0)) for i, (d, t, offs, _) in enumerate(CACHES[tags[0]][0])]
    test_ens = [(d, t, offs, np.mean([CACHES[g][1][i][3] for g in tags], axis=0)) for i, (d, t, offs, _) in enumerate(CACHES[tags[0]][1])]
    trans = np.mean([CACHES[g][2] for g in tags], axis=0)
    (delta, mode), sc = tune_decode(dev_ens, trans)
    print(f"ensemble of {tags}: o_delta {delta}, trim {mode} | dev strict P {sc['p']:.4f} R {sc['r']:.4f} F1 {sc['f1']:.4f}")
    print_scores(sc)
    print("saved", write_submission(decode_cache(test_ens, trans, delta, mode), "ensemble"))
    RESULTS["ensemble"] = {"P": sc["p"], "R": sc["r"], "F1": sc["f1"], "delta": delta, "trim": mode, "trainable_M": 0.0, "min": 0.0}

print(f"\n{'method':10s} {'devP':>7s} {'devR':>7s} {'devF1':>7s} {'trainM':>8s} {'min':>6s}")
for k, v in sorted(RESULTS.items(), key=lambda kv: -kv[1]["F1"]):
    print(f"{k:10s} {v['P']:7.4f} {v['R']:7.4f} {v['F1']:7.4f} {v['trainable_M']:8.2f} {v['min']:6.1f}")
if RESULTS:
    top = max(RESULTS, key=lambda k: RESULTS[k]["F1"])
    import shutil; shutil.copy(f"/kaggle/working/submission_{top}.zip", "/kaggle/working/submission_best.zip")
    print("best on dev:", top, "-> /kaggle/working/submission_best.zip   (dev is only", DEV_COUNT, "docs - treat gaps < 0.01 as noise)")


ensemble of ['houlsby', 'pfeiffer', 'prefix', 'prompt', 'bitfit', 'kd']: o_delta 0.5, trim 1 | dev strict P 0.5555 R 0.5135 F1 0.5215
  APP        P=0.259 R=0.085 F1=0.128  (tp 7, fp 20, fn 75)
  RESP       P=0.259 R=0.082 F1=0.125  (tp 7, fp 20, fn 78)
  A.COUNSEL  P=0.650 R=0.619 F1=0.634  (tp 13, fp 7, fn 8)
  R.COUNSEL  P=0.500 R=0.250 F1=0.333  (tp 3, fp 3, fn 9)
  JUDGE      P=0.647 R=0.524 F1=0.579  (tp 11, fp 6, fn 10)
  WIT        P=0.370 R=0.357 F1=0.364  (tp 10, fp 17, fn 18)
  AUTH       P=0.573 R=0.488 F1=0.527  (tp 59, fp 44, fn 62)
  COURT      P=0.883 R=0.934 F1=0.908  (tp 113, fp 15, fn 8)
  STAT       P=0.893 R=0.933 F1=0.913  (tp 251, fp 30, fn 18)
  PREC       P=0.425 R=0.567 F1=0.486  (tp 51, fp 69, fn 39)
  DATE       P=0.914 R=0.972 F1=0.942  (tp 171, fp 16, fn 5)
  CASENO     P=0.292 R=0.352 F1=0.319  (tp 19, fp 46, fn 35)
  MACRO      P=0.556 R=0.514 F1=0.521
saved /kaggle/working/submission_ensemble.zip

method        devP    devR   devF1   trainM    min
kd   